# Raw Data Download — ALPSS

Download raw experimental data files from the aimdl/datafiles API. This notebook shows how to query files by data type, download them in parallel, and organize them by sample (IGSN).

In [ ]:
import sys
from pathlib import Path
import requests

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from aimdl_examples import (
    get_girder_client,
    download_item_to_disk,
    paginate_datafiles,
    get_output_dir,
    fetch_and_write_run_metadata,
)

## Configuration

Select what data to download:

### Data Type

Choose one of:
- **`pdv_trace`** — Raw PDV velocity traces
- **`pdv_alpss_output`** — ALPSS analysis outputs (derivatives, fits, plots)
- **`pdv_experiment_log`** — Experiment metadata and settings

If downloading `pdv_alpss_output`, you can optionally filter by filename (see next cell).

In [ ]:
DATA_TYPE = "pdv_trace"

### Filename Filter (optional)

Only applies to `pdv_alpss_output`. Leave blank to download all, or choose one:
- `"velocity"` — Velocity curves
- `"velocity--smooth"` — Smoothed velocity
- `"noisefrac"`, `"voltage"`, `"veluncert"`, `"hel"`, `"iq"`, `"inputs"`, `"plots"`

The filter matches any file containing this string in its name.

In [ ]:
FILENAME_FILTER = ""

### Output Directory

Files are organized by sample ID (IGSN) in subdirectories. Files from the same sample go together.

In [ ]:
OUTPUT_DIR = f"./{DATA_TYPE}"
if DATA_TYPE == "pdv_alpss_output" and FILENAME_FILTER:
    OUTPUT_DIR = f"{OUTPUT_DIR}/{FILENAME_FILTER}"

## Download Files

This fetches run metadata (versions, run ID), then downloads all matching files in parallel using a thread pool.

In [ ]:
with requests.Session() as session:
    gc = get_girder_client(session=session)
    
    # Fetch and write run metadata
    fetch_and_write_run_metadata(gc, DATA_TYPE, OUTPUT_DIR)
    
    # Set up filter if downloading pdv_alpss_output with a filename filter
    item_filter = None
    if DATA_TYPE == "pdv_alpss_output" and FILENAME_FILTER:
        def item_filter(item):
            return FILENAME_FILTER in item.get("name", "")
    
    # Download files
    results = paginate_datafiles(
        gc,
        DATA_TYPE,
        worker_fn=lambda item: download_item_to_disk(
            gc,
            item,
            get_output_dir(item, OUTPUT_DIR)
        ),
        item_filter=item_filter
    )

In [ ]:
# Print summary
for result in results:
    print(f"Downloaded: {result['name']} ({result['size'] / 1024:.2f} KB)")
    
print(f"\nTotal files downloaded: {len(results)}")
print(f"Total size: {sum(r['size'] for r in results) / (1024**2):.2f} MB")

## Summary

View the downloaded files:

In [ ]:
import pandas as pd

df = pd.DataFrame(results)
print(f"Downloaded {len(df)} files of type '{DATA_TYPE}'")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nSample files:")
print(df[["name", "size"]].head(10))